In [ ]:
# load dataset test.pkl
import pickle
from PIL import Image
with open('data/test.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
Image.fromarray(data['trajectories'][0]['first_mask'])
# save image mask to jpg for testing
Image.fromarray(data['trajectories'][0]['first_mask']).save('test_mask.jpg')

In [ ]:
Image.fromarray(data['trajectories'][0]['first_mask'])

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image

def _save_image_with_timestamp(img_array, filepath, timestamp):
    """Save image with specific timestamp in EXIF data."""
    # Convert numpy array to PIL Image
    img = Image.fromarray(img_array)
    
    try:
        import piexif
        
        # Create EXIF data with correct timestamp tags
        exif_dict = {
            "0th": {
                piexif.ImageIFD.DateTime: timestamp.strftime("%Y:%m:%d %H:%M:%S"),
            },
            "Exif": {
                piexif.ExifIFD.DateTimeOriginal: timestamp.strftime("%Y:%m:%d %H:%M:%S"),
                piexif.ExifIFD.DateTimeDigitized: timestamp.strftime("%Y:%m:%d %H:%M:%S"),
            },
            "GPS": {},
            "1st": {},
            "thumbnail": None
        }
        
        # Convert to piexif format and save
        exif_bytes = piexif.dump(exif_dict)
        img.save(filepath, exif=exif_bytes)
        
    except ImportError:
        print("Warning: piexif not installed, saving without EXIF timestamp")
        print("Install with: pip install piexif")
        img.save(filepath)
        
        # Fallback: set file modification time
        import os
        timestamp_epoch = timestamp.timestamp()
        os.utime(filepath, (timestamp_epoch, timestamp_epoch))
        
    except Exception as e:
        print(f"Warning: Could not set EXIF timestamp: {e}")
        img.save(filepath)
        
        # Fallback: set file modification time
        import os
        timestamp_epoch = timestamp.timestamp()
        os.utime(filepath, (timestamp_epoch, timestamp_epoch))
def extract_trajectory_images(trajectories, output_dir="data/trajectory_images", base_timestamp=None):
    """
    Extract first and last images from trajectories and save as PNGs with ordered timestamps.
    
    :param trajectories: List of trajectory dictionaries
    :param output_dir: Directory to save images
    :param base_timestamp: Starting timestamp (datetime), defaults to now
    :returns: Number of images saved
    """
    from PIL import Image
    from PIL.ExifTags import IFD, TAGS
    from datetime import datetime, timedelta

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    if base_timestamp is None:
        base_timestamp = datetime(2025, 1, 1, 12, 0, 0)  # Fixed base date for consistency
    
    image_count = 0
    current_time = base_timestamp
    
    # Sort trajectories: bridge first, then fractal, by trajectory_id within each dataset
    sorted_trajs = sorted(trajectories, key=lambda x: (x['dataset'], x['trajectory_id']))
    
    for traj in sorted_trajs:
        dataset = traj['dataset']
        traj_id = traj['trajectory_id']
        
        # Save first image with timestamp
        first_img = traj['first_image']
        if first_img.dtype != np.uint8:
            first_img = (first_img * 255).astype(np.uint8)
        
        first_filename = f"first_image_{dataset}_{traj_id}.png"
        first_path = output_path / first_filename
        _save_image_with_timestamp(first_img, first_path, current_time)
        current_time += timedelta(minutes=1)  # Increment by 1 minute
        
        # Save last image with timestamp
        last_img = traj['last_image']
        if last_img.dtype != np.uint8:
            last_img = (last_img * 255).astype(np.uint8)
            
        last_filename = f"last_image_{dataset}_{traj_id}.png"
        last_path = output_path / last_filename
        _save_image_with_timestamp(last_img, last_path, current_time)
        current_time += timedelta(minutes=1)  # Increment by 1 minute
        
        image_count += 2
    
    print(f"Saved {image_count} images with ordered timestamps to {output_path}")
    print(f"Order: first_image_0, last_image_0, first_image_1, last_image_1, ...")
    print(f"Datasets ordered: bridge first, then fractal")


In [ ]:
import pickle
with open('data/bridge_fractal_n300.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
extract_trajectory_images(data['trajectories'], "data/bridge_fractal_n300_images", base_timestamp=None)

#### Load Inpaintings made with Google Magic Editor

In [ ]:
import pickle
from PIL import Image
import numpy as np
from pathlib import Path

with open('data/bridge_fractal_n300.pkl', 'rb') as f:
    data = pickle.load(f)

# Check structure of trajectory data
print("Data keys:", data.keys())
print("Number of trajectories:", len(data['trajectories']))
print("First trajectory keys:", data['trajectories'][0].keys())
print("First trajectory dataset:", data['trajectories'][0]['dataset'])
print("First trajectory ID:", data['trajectories'][0]['trajectory_id'])
print("First image shape:", data['trajectories'][0]['first_image'].shape)

In [ ]:
def load_inpainted_dataset_magic_editor(original_pkl_path, inpainted_images_dir, output_pkl_path=None):
    """
    Load original dataset and add inpainted images (that are stored in specific directory as png files) as new keys.
    From Inpaintings created with Google Magic Editor

    :param original_pkl_path: Path to original pickle file
    :param inpainted_images_dir: Directory containing inpainted images
    :param output_pkl_path: Optional path to save merged dataset
    :returns: Dataset dict with original and inpainted images
    """
    
    # Load original dataset
    with open(original_pkl_path, 'rb') as f:
        data = pickle.load(f)
    
    inpainted_dir = Path(inpainted_images_dir)
    if not inpainted_dir.exists():
        raise FileNotFoundError(f"Inpainted images directory not found: {inpainted_dir}")
    
    print(f"Loading inpainted images from: {inpainted_dir}")
    
    # Add inpainted images to each trajectory
    missing_files = []
    loaded_count = 0
    
    for traj in data['trajectories']:
        dataset = traj['dataset']
        traj_id = traj['trajectory_id']
        
        # Load inpainted first image
        first_inpainted_path = inpainted_dir / f"first_image_{dataset}_{traj_id}.png"
        if first_inpainted_path.exists():
            first_inpainted = np.array(Image.open(first_inpainted_path))
            traj['first_image_inpainted'] = first_inpainted
            loaded_count += 1
        else:
            missing_files.append(str(first_inpainted_path))
            traj['first_image_inpainted'] = None
        
        # Load inpainted last image  
        last_inpainted_path = inpainted_dir / f"last_image_{dataset}_{traj_id}.png"
        if last_inpainted_path.exists():
            last_inpainted = np.array(Image.open(last_inpainted_path))
            traj['last_image_inpainted'] = last_inpainted
            loaded_count += 1
        else:
            missing_files.append(str(last_inpainted_path))
            traj['last_image_inpainted'] = None
    
    print(f"Loaded {loaded_count} inpainted images")
    if missing_files:
        print(f"Warning: {len(missing_files)} inpainted images not found:")
        for missing in missing_files[:10]:  # Show first 10 missing files
            print(f"  {missing}")
        if len(missing_files) > 10:
            print(f"  ... and {len(missing_files) - 10} more")
    
    # Save merged dataset if output path provided
    if output_pkl_path:
        with open(output_pkl_path, 'wb') as f:
            pickle.dump(data, f)
        print(f"Saved merged dataset to: {output_pkl_path}")
    
    return data

In [ ]:
# Example usage - uncomment when you have inpainted images ready
merged_data = load_inpainted_dataset_magic_editor(
    original_pkl_path='data/bridge_fractal_n300.pkl',
    inpainted_images_dir='data/bridge_fractal_n300_images_inpainted',
    # output_pkl_path='data/bridge_fractal_n300_with_inpainted.pkl'
)

# Verify the structure
print("Keys in first trajectory after merging:")
print(list(merged_data['trajectories'][0].keys()))

#### Plot Trajectory Comparison

In [ ]:
from plot_inpaintings import plot_original_inpainting_comparison

In [ ]:
# Example with bridge_fractal_n300 dataset (without inpainted images for now)
plot_original_inpainting_comparison(merged_data, trajectory_idx=0, save_path=None)

In [ ]:
from plot_inpaintings import plot_multiple_inpainted_trajectories

In [ ]:
# Example: Plot first 3 trajectories
plot_multiple_inpainted_trajectories(merged_data, num_trajectories=3, ) # save_dir="data/trajectory_plots"

#### Interactive HTML Trajectory Viewer

In [ ]:
from create_traj_viewer import create_interactive_viewer

In [ ]:
# Create interactive viewer with a subset of trajectories for testing
# Use subset_size to limit the number of trajectories (useful for large datasets)
viewer_path = create_interactive_viewer(
    merged_data, 
    output_path="trajectory_viewer_with_data.html",
    template_path="trajectory_viewer.html",
    subset_size=20  # Start with 20 trajectories for testing
)